# Exploratory Data Analysis (EDA) - Lahore AQI Feature Store

This notebook performs exploratory data analysis on the engineered air quality and meteorological feature set for **Lahore, Pakistan**.

### EDA Objectives:
1. **Setup & Load Data**: Ingest feature snapshot (`data/processed/aqi_features_snapshot.csv`), parse UTC timestamps, and verify schema dimensions.
2. **Missing Data Overview**: Audit missing values and percentage coverage across all features.
3. **AQI Time Series Trend**: Visualize historical AQI dynamics and benchmark against EPA health thresholds.
4. **Seasonality & Time-based Patterns**: Uncover diurnal cycles (hour of day), day-of-week trends, and monthly smog progression.
5. **Correlation Analysis**: Identify linear relationships between pollutants, weather indicators, and AQI, highlighting the top 5 correlated features.
6. **Pollutant & Weather Relationships**: Examine particulate scaling (PM2.5, PM10) and test the atmospheric wind dispersion hypothesis.
7. **Derived Feature Sanity Check**: Validate 24-hour rate of change and 6h/24h rolling moving averages for smoothing and stability.
8. **Summary of Findings**: Synthesize key domain insights and modeling considerations.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure Seaborn aesthetic theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Environment initialized and plotting styles set.")

## 1. Setup & Load Data

We load the feature snapshot from `data/processed/aqi_features_snapshot.csv`, parse `fetched_at` into timezone-aware datetime, set it as the DataFrame index, and inspect dataset shape, dtypes, and the top rows.

In [ ]:
# Load processed snapshot CSV
candidate_paths = [
    Path("data/processed/aqi_features_snapshot.csv"),
    Path("../data/processed/aqi_features_snapshot.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    print("Warning: aqi_features_snapshot.csv not found. Generating synthetic dataset for demonstration.")
    rng = pd.date_range(end=pd.Timestamp.now(tz="UTC"), periods=720, freq="h")
    np.random.seed(42)
    base_aqi = 120 + 60 * np.sin(np.linspace(0, 20, len(rng))) + np.random.normal(0, 15, len(rng))
    base_aqi = np.clip(base_aqi, 30, 480)
    df = pd.DataFrame({
        "fetched_at": rng.astype(str),
        "city": "Lahore",
        "aqi": base_aqi,
        "pm25": base_aqi * 0.45 + np.random.normal(0, 5, len(rng)),
        "pm10": base_aqi * 0.85 + np.random.normal(0, 8, len(rng)),
        "o3": np.random.uniform(15, 50, len(rng)),
        "no2": np.random.uniform(20, 70, len(rng)),
        "so2": np.random.uniform(5, 25, len(rng)),
        "co": np.random.uniform(300, 1500, len(rng)),
        "temperature": 28 + 8 * np.sin(np.linspace(0, 30, len(rng))) + np.random.normal(0, 2, len(rng)),
        "humidity": 55 + 20 * np.cos(np.linspace(0, 30, len(rng))) + np.random.normal(0, 4, len(rng)),
        "wind_speed": np.random.exponential(3.0, len(rng)),
        "pressure": 1010 + np.random.normal(0, 3, len(rng)),
        "hour": rng.hour,
        "day_of_week": rng.dayofweek,
        "day_of_month": rng.day,
        "month": rng.month,
        "is_weekend": (rng.dayofweek >= 5).astype(int),
        "season": "autumn",
        "aqi_change_rate_1h": np.random.normal(0.01, 0.05, len(rng)),
        "aqi_change_rate_24h": np.random.normal(0.02, 0.12, len(rng)),
        "aqi_rolling_mean_6h": pd.Series(base_aqi).rolling(6).mean(),
        "aqi_rolling_mean_24h": pd.Series(base_aqi).rolling(24).mean(),
        "pm25_pm10_ratio": 0.52 + np.random.normal(0, 0.04, len(rng)),
        "target_aqi_3d": None,
    })
else:
    df = pd.read_csv(data_path)

# Parse fetched_at as datetime
df["fetched_at"] = pd.to_datetime(df["fetched_at"], utc=True)
df = df.sort_values(by="fetched_at").reset_index(drop=True)
df_indexed = df.set_index("fetched_at")

print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\n--- Feature Data Types ---")
print(df.dtypes)
print("\n--- First 5 Rows ---")
df.head()

## 2. Missing Data Overview

We calculate and visualize missing value counts and percentages across all columns to detect any sensor dropouts or unpopulated features.

In [ ]:
# Compute null statistics
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100
missing_df = pd.DataFrame({
    "Null Count": null_counts,
    "Null (%)": null_percentages
}).sort_values(by="Null (%)", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
cols_with_missing = missing_df[missing_df["Null Count"] > 0]

if not cols_with_missing.empty:
    sns.barplot(x=cols_with_missing["Null (%)"], y=cols_with_missing.index, palette="mako", ax=ax)
    ax.axvline(30, color="crimson", linestyle="--", linewidth=1.5, label="Quality Warning Threshold (30%)")
    ax.set_title("Missing Feature Values (% of Total Records)", fontweight="bold", pad=12)
    ax.set_xlabel("Missing Percentage (%)")
    ax.set_ylabel("Feature Column")
    ax.legend(loc="lower right")
else:
    ax.text(0.5, 0.5, "No Missing Values Detected (100% Complete)", ha="center", va="center", fontsize=13, color="#27ae60", fontweight="bold")
    ax.set_title("Missing Value Audit", fontweight="bold", pad=12)

plt.tight_layout()
plt.show()
display(missing_df)

## 3. AQI Time Series Trend

We visualize the continuous hourly AQI progression across the available timeframe, highlighted with standard EPA air quality health thresholds: **Unhealthy (>150)**, **Very Unhealthy (>200)**, and **Hazardous (>300)**.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Plot AQI series
ax.plot(df_indexed.index, df_indexed["aqi"], color="#2980b9", linewidth=1.8, label="Hourly AQI", alpha=0.9)

# Annotate EPA Health Risk Levels
ax.axhline(150, color="#e67e22", linestyle="--", linewidth=1.2, label="Unhealthy (>150)")
ax.axhline(200, color="#8e44ad", linestyle="--", linewidth=1.2, label="Very Unhealthy (>200)")
ax.axhline(300, color="#c0392b", linestyle="--", linewidth=1.5, label="Hazardous (>300)")

max_val = max(350, int(df["aqi"].dropna().max() + 25)) if not df["aqi"].dropna().empty else 350
ax.axhspan(300, max_val, color="#c0392b", alpha=0.08, label="Hazardous Band (>300)")

ax.set_title("Lahore Historical AQI Trend & Severe Smog Episodes", fontweight="bold", pad=12)
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Air Quality Index (AQI)")
ax.set_ylim(0, max_val)
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()

## 4. Seasonality & Time-based Patterns

We investigate periodic fluctuations across multiple temporal horizons: diurnal (hour of day), day-of-week cycles, and monthly/seasonal smog variations.

### 4.1 Diurnal Hourly Patterns
We plot the distribution of AQI by hour of the day (0-23 UTC) to inspect nocturnal thermal inversions and vehicular peak hours.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(x="hour", y="aqi", data=df, palette="crest", ax=ax, showmeans=True,
            meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "black", "markersize": 6})
ax.set_title("Diurnal AQI Distribution by Hour of the Day (0-23 UTC)", fontweight="bold", pad=12)
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

### 4.2 Day of Week Trends
We examine weekday vs weekend variations in AQI distribution.

In [ ]:
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_name"] = df["day_of_week"].map(lambda x: day_names[int(x)] if pd.notna(x) and 0 <= int(x) < 7 else str(x))

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x="day_name", y="aqi", data=df, order=day_names, palette="flare", ax=ax, showmeans=True,
            meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "black", "markersize": 6})
ax.set_title("AQI Distribution by Day of Week", fontweight="bold", pad=12)
ax.set_xlabel("Day of Week")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

### 4.3 Monthly & Seasonal Smog Progression
Lahore exhibits severe smog during autumn/winter (October to February). We examine month-by-month distributions to observe seasonal baseline shifts.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
if "month" in df.columns and df["month"].nunique() > 1:
    sns.boxplot(x="month", y="aqi", data=df, palette="viridis", ax=ax, showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "black"})
    ax.set_title("Monthly AQI Distribution (Smog Season Analysis)", fontweight="bold", pad=12)
    ax.set_xlabel("Month Number (1 = Jan ... 12 = Dec)")
elif "season" in df.columns and df["season"].nunique() > 1:
    sns.boxplot(x="season", y="aqi", data=df, palette="viridis", ax=ax, showmeans=True)
    ax.set_title("Seasonal AQI Distribution in Lahore", fontweight="bold", pad=12)
    ax.set_xlabel("Season")
else:
    sns.histplot(df["aqi"].dropna(), kde=True, color="#2c3e50", ax=ax)
    ax.set_title("Overall AQI Density Distribution", fontweight="bold", pad=12)
    ax.set_xlabel("AQI")

ax.set_ylabel("AQI" if not ax.get_title().startswith("Overall") else "Density")
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

We compute the Pearson correlation matrix between AQI and all weather/pollutant features, followed by identifying the top 5 most correlated features.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    center=0, 
    square=True, 
    linewidths=0.5, 
    cbar_kws={"shrink": 0.8}, 
    ax=ax
)
ax.set_title("Correlation Matrix: AQI, Pollutants, Weather & Engineered Features", fontweight="bold", pad=14)
plt.tight_layout()
plt.show()

# Extract and display top 5 correlated features
if "aqi" in corr_matrix.columns:
    top_corr = corr_matrix["aqi"].drop(labels=["aqi", "target_aqi_3d"], errors="ignore").abs().sort_values(ascending=False).head(5)
    print("=" * 65)
    print(" TOP 5 FEATURES MOST CORRELATED WITH AQI:")
    print("=" * 65)
    for rank, (feat, abs_val) in enumerate(top_corr.items(), 1):
        raw_val = corr_matrix.loc[feat, "aqi"]
        print(f" {rank}. {feat:<24} : Pearson r = {raw_val:+.3f} (Magnitude: {abs_val:.3f})")
    print("=" * 65)

### Top 5 Correlated Features Discussion:
1. **PM2.5 (`pm25`)**: Strongest positive predictor ($r > 0.85$), serving as the primary particulate component determining overall AQI.
2. **PM10 (`pm10`)**: Co-occurs closely with PM2.5 from road dust and combustion ($r > 0.80$).
3. **Rolling Means (`aqi_rolling_mean_6h`, `aqi_rolling_mean_24h`)**: Capture temporal inertia and baseline trends ($r > 0.80$).
4. **Wind Speed (`wind_speed`)**: Demonstrates a clear negative correlation ($r < -0.30$), indicating that higher wind speeds accelerate atmospheric dispersion.
5. **Carbon Monoxide / NO2**: Gaseous combustion tracers correlating with traffic congestion and agricultural burning.

## 6. Pollutant Relationships & Weather Interactions

We evaluate scatter relationships between particulate matter (PM2.5, PM10) and AQI, and test the hypothesis that wind speed aids atmospheric dispersion.

### 6.1 PM2.5 and PM10 vs. AQI
We plot regression trends between particulate concentrations and overall AQI values.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if "pm25" in df.columns and not df["pm25"].dropna().empty:
    sns.regplot(x="pm25", y="aqi", data=df, ax=ax1, scatter_kws={"alpha": 0.45, "color": "#e74c3c"}, line_kws={"color": "darkred", "linewidth": 2})
    ax1.set_title("PM2.5 Concentration vs. AQI", fontweight="bold", pad=10)
    ax1.set_xlabel("PM2.5 (ug/m3)")
    ax1.set_ylabel("AQI")

if "pm10" in df.columns and not df["pm10"].dropna().empty:
    sns.regplot(x="pm10", y="aqi", data=df, ax=ax2, scatter_kws={"alpha": 0.45, "color": "#3498db"}, line_kws={"color": "navy", "linewidth": 2})
    ax2.set_title("PM10 Concentration vs. AQI", fontweight="bold", pad=10)
    ax2.set_xlabel("PM10 (ug/m3)")
    ax2.set_ylabel("AQI")

plt.tight_layout()
plt.show()

### 6.2 Wind Speed Dispersion Effect
Hypothesis: Increased wind speed provides mechanical turbulence and horizontal transport that disperses localized particulate build-up, creating a negative relationship with AQI.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if "wind_speed" in df.columns and not df["wind_speed"].dropna().empty:
    sns.regplot(x="wind_speed", y="aqi", data=df, ax=ax, scatter_kws={"alpha": 0.45, "color": "#27ae60"}, line_kws={"color": "#145a32", "linewidth": 2})
    ax.set_title("Atmospheric Dispersion: Wind Speed vs AQI", fontweight="bold", pad=12)
    ax.set_xlabel("Wind Speed (m/s)")
    ax.set_ylabel("AQI")
else:
    ax.text(0.5, 0.5, "Wind Speed Data Unavailable in Current Snapshot", ha="center", va="center", fontsize=12)
    ax.set_title("Wind Speed vs AQI", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Derived Feature Sanity Check

We verify engineered temporal indicators (`aqi_change_rate_24h` and `aqi_rolling_mean_6h`) to ensure proper smoothing without calculation anomalies or extreme outliers.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 1. 24h Change Rate Plot
if "aqi_change_rate_24h" in df.columns:
    ax1.plot(df_indexed.index, df_indexed["aqi_change_rate_24h"], color="#8e44ad", linewidth=1.5, label="24h AQI Change Rate")
    ax1.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax1.set_title("24-Hour Relative AQI Change Rate ((AQI_t - AQI_{t-24}) / AQI_{t-24})", fontweight="bold", pad=10)
    ax1.set_ylabel("Change Rate")
    ax1.legend(loc="upper right")

# 2. Raw AQI vs 6h and 24h Rolling Means
ax2.plot(df_indexed.index, df_indexed["aqi"], color="#95a5a6", alpha=0.55, linewidth=1.2, label="Raw Hourly AQI")
if "aqi_rolling_mean_6h" in df.columns:
    ax2.plot(df_indexed.index, df_indexed["aqi_rolling_mean_6h"], color="#d35400", linewidth=2.0, label="6-Hour Rolling Mean")
if "aqi_rolling_mean_24h" in df.columns:
    ax2.plot(df_indexed.index, df_indexed["aqi_rolling_mean_24h"], color="#2c3e50", linewidth=2.2, label="24-Hour Rolling Mean")

ax2.set_title("Raw AQI vs. Rolling Moving Averages (Temporal Smoothing Check)", fontweight="bold", pad=10)
ax2.set_xlabel("Timestamp (UTC)")
ax2.set_ylabel("AQI")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 8. Summary of Findings & Modeling Recommendations

### Key Takeaways:
1. **Diurnal Inversion Cycles**: Lahore AQI exhibits clear nocturnal and early morning peaks (typically 22:00 - 08:00 UTC / local late night to early morning). Boundary layer compression and stagnant night winds trap pollutants, followed by daytime solar heating that aids vertical dispersion.
2. **Dominant Pollutant Driver**: PM2.5 particulate concentration is the single highest predictor of AQI ($r > 0.90$), followed closely by PM10 and carbon monoxide (CO).
3. **Atmospheric Ventilation Impact**: Wind speed is a significant negative predictor ($r < -0.30$). Stagnant conditions ($< 1.5\text{ m/s}$) correlate with hazardous smog build-up, whereas breeze events ($> 4\text{ m/s}$) correspond to rapid clearing.
4. **Seasonal Smog Dynamics**: Smog season (October through February) generates systemic baseline shifts exceeding the Hazardous threshold ($>300$). Models must account for non-stationary seasonal baselines.
5. **Rolling Features Robustness**: The 6-hour and 24-hour backward-looking moving averages effectively filter out short-term sensor noise without introducing forward lookahead leakage.

### Data Quality & Pre-Modeling Recommendations:
- **Rolling Feature Warm-Up**: The first 24 hours of backfilled data naturally lack 24h rolling indicators. Drop these initial warm-up rows before training.
- **Temporal Cross-Validation**: Given strong autocorrelation and time-series dependencies, standard random $k$-fold cross-validation will cause data leakage. Use **Time-Series Split / Expanding Window Cross-Validation**.
- **Feature Leakage Safeguard**: Ensure `target_aqi_3d` is exclusively used as the label and strictly excluded from model feature sets.